In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from piepy.viz.plots import psychometric, reaction_time_cloud, reaction_time_dist

DATA_ROOT = Path(sys.executable).parents[2] / "data"
DATA_ROOT

In [ ]:
# reading the whole data
all_data = pl.read_parquet(f"{DATA_ROOT}/260923_training_data.parquet")


# filter for subset of sessins that are in the Cetindag et al 2026 manuscript
sessions = [
            "230227_KC139_detect__no_cam_KC", 
            "230120_KC143_detect__no_cam_KC", 
            "230403_KC144_detect__no_cam_KC", 
            "230609_KC145_detect__no_cam_KC", 
            "230511_KC146_detect__no_cam_KC", 
            "240506_KC147_detect__no_cam_KC", 
            "240429_KC148_detect__no_cam_KC", 
            "240424_KC149_detect__no_cam_KC", 
            "240814_KC151_detect__no_cam_KC", 
            "240810_KC152_detect__no_cam_KC", 
            # below animals can be excluded
            "220915_KC300_detect__no_cam_KC", 
            "220928_KC301_detect__no_cam_KC", 
            "240718_KC302_detect__no_cam_KC", 
            "240711_KC303_detect__no_cam_KC", 
            "240807_KC304_detect__no_cam_KC", 
            ]

subset_data = all_data.filter(pl.col("sessiondir").is_in(sessions))
subset_data

In [ ]:
early_data = subset_data.filter(pl.col("outcome")=="early")
early_data["contrast"].unique()

In [ ]:
early_data = subset_data.filter(pl.col("outcome")=="early")
catch_data = subset_data.filter((pl.col("outcome")!="early") & (pl.col("isCatch")))

# early_and_catch_data = pl.concat([early_data,catch_data]).sort("total_trial_no")

pr = reaction_time_dist(data=early_data,
                        value="reaction_time",
                        bin_width=10,
                        average_over=None,
                        label="early",
                        )

print(pr.data)

ax = pr.figure[1]

pr = reaction_time_dist(data=catch_data,
                        value="reaction_time",
                        bin_width=10,
                        average_over=None,
                        ax=ax,
                        color="#000000",
                        label="Catch",
                        )


ax.set_xlim([-5000,2000])
ax.set_ylim([0,100])

In [ ]:
pr = reaction_time_dist(data=subset_data,
                        value="duration_blank",
                        bin_width=10,
                        average_over=None,
                        )

pr.figure[1].set_title("distribution of wait time durations",fontsize=20)

In [ ]:
subset_data = subset_data.with_columns((pl.col("t_trialinit") - pl.col("t_vstimstart")).alias("stim_start_time"))

In [ ]:
pr = reaction_time_dist(data=subset_data,
                        value="stim_start_time",
                        bin_width=10,
                        average_over=None,
                        )